# 🎤 NVIDIA Canary-Qwen-2.5B 語音轉文字轉錄器
**SOTA 英語 ASR** — WER 5.63% | FastConformer + Qwen3-1.7B | **CTC Forced Alignment 逐字時間戳**

⚠️ **English-only** · T4 (免費) 可跑 · 第一步安裝約 3-8 分鐘 · 第一步完成後會自動重啟 Runtime，重啟後直接跑第二步

In [ ]:
# @title 🛠️ 第一步：安裝環境 (加速版 — 完成後自動重啟 Runtime)
import subprocess, sys, os, time, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed

t_total = time.time()

def run(cmd, desc, shell=False):
    t0 = time.time()
    r = subprocess.run(cmd, shell=shell, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    el = time.time() - t0
    if r.returncode != 0:
        print(f'  ❌ {desc} ({el:.0f}s)')
        for line in r.stdout.strip().split('\n')[-15:]: print(f'     {line}')
        raise RuntimeError(desc)
    return desc, el

print('📦 加速安裝中...\n')

# === Phase 1: 並行 — apt + uv + Python 前置依賴同時跑 ===
print('  ⚡ Phase 1: 並行安裝基礎依賴...')
with ThreadPoolExecutor(max_workers=3) as pool:
    futs = [
        pool.submit(run, 'apt-get -qq -y update && apt-get -qq -y install ffmpeg libsndfile1',
                    'ffmpeg + libsndfile', True),
        pool.submit(run, [sys.executable,'-m','pip','install','-q','uv'],
                    'uv (Rust 加速器)'),
        pool.submit(run, [sys.executable,'-m','pip','install','-q',
                    'Cython','packaging','pydub','soundfile','huggingface_hub'],
                    'Python 依賴'),
    ]
    for f in as_completed(futs):
        desc, el = f.result()
        print(f'     ✅ {desc} ({el:.0f}s)')

# === Phase 2: NeMo — shallow clone + uv 安裝 ===
print('\n  ⚡ Phase 2: NeMo (shallow clone + uv)...')
t1 = time.time()

# shallow clone (--depth 1 比 pip 的 full clone 快很多)
nemo_dir = '/tmp/nemo_src'
if os.path.exists(nemo_dir): shutil.rmtree(nemo_dir)
run(['git','clone','--depth','1','--single-branch','--branch','main',
     'https://github.com/NVIDIA-NeMo/NeMo.git', nemo_dir],
    'git clone (shallow)')

# 用 uv pip install 取代 pip — dependency resolution 快 10-100x
uv = shutil.which('uv') or os.path.expanduser('~/.local/bin/uv')
if os.path.exists(uv):
    run([uv,'pip','install','--system','-q',f'{nemo_dir}[asr]'],
        f'NeMo [asr] via uv')
else:
    # fallback to pip
    print('     ⚠️ uv 不可用，fallback to pip')
    run([sys.executable,'-m','pip','install','-q',f'{nemo_dir}[asr]'],
        'NeMo [asr] via pip')
print(f'     ✅ NeMo 安裝完成 ({time.time()-t1:.0f}s)')

# === Phase 3: NumPy ABI 修復 ===
print('\n  ⚡ Phase 3: NumPy ABI 對齊...')
pip_cmd = [uv,'pip','install','--system'] if os.path.exists(uv) else [sys.executable,'-m','pip','install']
run([sys.executable,'-m','pip','uninstall','-y','numpy','scipy','numba','librosa','llvmlite'],
    '移除舊版')
run(pip_cmd + ['-q','--force-reinstall','--no-cache-dir','numpy','scipy','numba','librosa'],
    '重建 ABI')

# === 驗證 ===
r = subprocess.run([sys.executable,'-c',
    'from nemo.collections.speechlm2.models import SALM; print("  ✅ speechlm2.SALM 可用")'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print(r.stdout.strip() if r.returncode==0 else f'  ❌ {r.stderr.strip().split(chr(10))[-1]}')

# 清理
shutil.rmtree(nemo_dir, ignore_errors=True)

elapsed = time.time() - t_total
print(f'\n✅ 完成！總耗時 {elapsed:.0f}s ({elapsed/60:.1f} 分鐘)')
print('   3 秒後自動重啟 Runtime → 重啟後跑【第二步】')
time.sleep(3)
os.kill(os.getpid(), 9)


In [ ]:
# @title 🚀 第二步：上傳音檔 → 轉錄
import os, sys, subprocess, warnings, time, gc, threading, logging
import re, inspect, tempfile
from typing import Tuple, List
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

# === 全域靜音 (包含 NeMo INFO + nv_one_logger) ===
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['NEMO_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)  # 全域壓掉 WARNING 及以下
# NeMo INFO logs 需要在 import 前就設定 root logger
logging.basicConfig(level=logging.ERROR, force=True)

import torch
import numpy as np
import soundfile as sf
import torchaudio
from pydub import AudioSegment

# 再次確保所有 noisy loggers 都被壓掉
for _n in ['nemo_logger','nemo','transformers','huggingface_hub',
           'huggingface_hub.utils._http','filelock','fsspec',
           'nv_one_logger','nv_one_logger.api.config',
           'nv_one_logger.training_telemetry.api.training_telemetry_provider',
           'nemo.collections','nemo.utils']:
    logging.getLogger(_n).setLevel(logging.CRITICAL)

# === Qwen3 Patch (v3.2) ===
import transformers, torch.nn as _nn
_DUMMY = _nn.Embedding(1, 1)
for _cp in ['transformers.models.qwen3.modeling_qwen3.Qwen3Model',
            'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM']:
    try:
        _mp, _cn = _cp.rsplit('.', 1)
        _cls = getattr(__import__(_mp, fromlist=[_cn]), _cn)
        if not hasattr(_cls, '_p'):
            def _mg(d=_DUMMY):
                def g(self):
                    if hasattr(self,'embed_tokens'): return self.embed_tokens
                    m=getattr(self,'model',None)
                    if m and hasattr(m,'embed_tokens'): return m.embed_tokens
                    return d
                return g
            def _ms():
                def s(self,v):
                    if hasattr(self,'embed_tokens'): self.embed_tokens=v
                    elif hasattr(self,'model') and hasattr(self.model,'embed_tokens'): self.model.embed_tokens=v
                return s
            _cls.get_input_embeddings=_mg(); _cls.set_input_embeddings=_ms(); _cls._p=True
    except: pass
if not hasattr(transformers.PreTrainedModel, '_p'):
    _og = transformers.PreTrainedModel.get_input_embeddings
    def _sg(self, _d=_DUMMY):
        try: return _og(self)
        except (NotImplementedError, AttributeError):
            if hasattr(self,'embed_tokens'): return self.embed_tokens
            m=getattr(self,'model',None)
            if m and hasattr(m,'embed_tokens'): return m.embed_tokens
            return _d
    transformers.PreTrainedModel.get_input_embeddings=_sg; transformers.PreTrainedModel._p=True

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# === 資料結構 ===
def _fmt(t, srt=False):
    t=max(0.0,t); h,r=divmod(int(t),3600); m,s=divmod(r,60)
    return f'{h:02d}:{m:02d}:{s:02d},{int((t-int(t))*1000):03d}' if srt else f'{h:02d}:{m:02d}:{s:02d}'

@dataclass
class Word:
    s: float; e: float; text: str; score: float = 1.0

@dataclass
class Seg:
    s: float; e: float; text: str; words: list = None
    def line(self): return f'[{_fmt(self.s)} → {_fmt(self.e)}] {self.text}'
    def srt(self, i): return f'{i}\n{_fmt(self.s,True)} --> {_fmt(self.e,True)}\n{self.text}\n'

def dedup(prev, curr, n=10):
    if not prev or not curr: return curr
    wp,wc = prev.split(), curr.split()
    f=lambda w: re.sub(r'[^\w\s]','',w).lower()
    a,b = [f(w) for w in wp], [f(w) for w in wc]
    k=0
    for i in range(1, min(len(a),len(b),n)+1):
        if a[-i:]==b[:i]: k=i
    return ' '.join(wc[k:]) if k else curr

# === 音訊 ===
def convert_audio(src):
    fd,wav=tempfile.mkstemp(suffix='.wav'); os.close(fd)
    try:
        subprocess.run(['ffmpeg','-y','-threads','2','-i',src,'-ar','16000','-ac','1','-c:a','pcm_s16le','-loglevel','error',wav],check=True,capture_output=True)
    except:
        AudioSegment.from_file(src).set_frame_rate(16000).set_channels(1).set_sample_width(2).export(wav,format='wav')
    return wav, sf.info(wav).duration

def split_audio(wav, dur, maxs=35.0, overlap=1.0):
    if dur <= maxs+5: return [(wav,0.0,dur)]
    data,sr = sf.read(wav)
    chunks,s = [],0.0
    while s<dur:
        e=min(s+maxs,dur); fd,cp=tempfile.mkstemp(suffix='.wav'); os.close(fd)
        sf.write(cp,data[int(s*sr):int(e*sr)],sr); chunks.append((cp,s,e)); s+=maxs-overlap
    return chunks

# === CTC Forced Alignment (torchaudio) ===
class CTCAligner:
    """用 torchaudio MMS forced alignment 產生逐字時間戳。"""
    def __init__(self, device='cpu'):
        self.device = device
        self.model = None
        self.labels = None
        self.sr = 16000

    def _load(self):
        if self.model is not None: return
        import torchaudio
        bundle = torchaudio.pipelines.MMS_FA
        self.model = bundle.get_model().to(self.device).eval()
        self.labels = bundle.get_labels()
        self.label2idx = {l:i for i,l in enumerate(self.labels)}
        self.sr = bundle.sample_rate  # 16000
        # MMS blank/star tokens
        self.blank_idx = self.label2idx.get('*', self.label2idx.get('<blank>', 0))

    @torch.inference_mode()
    def align(self, wav_path: str, transcript: str, offset: float = 0.0) -> list:
        """對單段音檔做 forced alignment，回傳 [Word(...), ...]"""
        import torchaudio
        self._load()

        waveform, sr = torchaudio.load(wav_path)
        if sr != self.sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.sr)
        waveform = waveform.to(self.device)

        # 取得 emission (log probs)
        emission, _ = self.model(waveform)
        emission = emission[0].cpu()  # (T, C)

        # 清理 transcript → 只保留 MMS 支援的字元
        clean = transcript.lower().strip()
        tokens = []
        clean_chars = []
        for ch in clean:
            if ch in self.label2idx:
                tokens.append(self.label2idx[ch])
                clean_chars.append(ch)
            elif ch == ' ':
                # MMS 用 '|' 代表 word boundary
                if '|' in self.label2idx:
                    tokens.append(self.label2idx['|'])
                    clean_chars.append('|')
        if not tokens:
            return []

        token_t = torch.tensor([tokens], dtype=torch.int32)
        # torchaudio forced_align
        try:
            aligned, scores = torchaudio.functional.forced_align(
                emission.unsqueeze(0), token_t, blank=self.blank_idx
            )
            aligned = aligned[0]  # (T,)
            scores = scores[0]
        except Exception:
            return []

        # 從 frame-level alignment 提取 token spans
        ratio = waveform.shape[1] / emission.shape[0] / self.sr  # seconds per frame
        token_spans = []
        i = 0
        while i < len(aligned):
            if aligned[i] > 0:  # non-blank
                tok_idx = aligned[i].item()
                start_frame = i
                while i < len(aligned) and aligned[i] == tok_idx:
                    i += 1
                end_frame = i
                token_spans.append((start_frame, end_frame, tok_idx, scores[start_frame:end_frame].mean().item()))
            else:
                i += 1

        # 將 token spans 合併成 words (以 '|' 分隔)
        words = []
        current_word_chars = []
        word_start = None
        word_end = None
        word_scores = []

        for start_f, end_f, tok_idx, score in token_spans:
            char = self.labels[tok_idx] if tok_idx < len(self.labels) else '?'
            if char == '|':
                # word boundary
                if current_word_chars:
                    words.append(Word(
                        s=offset + word_start * ratio,
                        e=offset + word_end * ratio,
                        text=''.join(current_word_chars),
                        score=sum(word_scores)/len(word_scores) if word_scores else 0.0
                    ))
                    current_word_chars = []
                    word_start = None
                    word_scores = []
                continue
            if word_start is None:
                word_start = start_f
            word_end = end_f
            current_word_chars.append(char)
            word_scores.append(score)

        # 最後一個 word
        if current_word_chars:
            words.append(Word(
                s=offset + word_start * ratio,
                e=offset + word_end * ratio,
                text=''.join(current_word_chars),
                score=sum(word_scores)/len(word_scores) if word_scores else 0.0
            ))

        return words

    def align_segments(self, wav_path: str, segs: list) -> list:
        """對已有的 Seg list 做 forced alignment，回填逐字時間戳。"""
        import torchaudio
        self._load()

        waveform, sr = torchaudio.load(wav_path)
        if sr != self.sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.sr)
        total_samples = waveform.shape[1]

        aligned_segs = []
        for seg in segs:
            # 切出這個 seg 對應的音訊區間
            s_sample = int(seg.s * self.sr)
            e_sample = min(int(seg.e * self.sr), total_samples)
            if e_sample <= s_sample:
                aligned_segs.append(seg)
                continue

            chunk = waveform[:, s_sample:e_sample]
            # 存暫存檔
            fd, tmp = tempfile.mkstemp(suffix='.wav')
            os.close(fd)
            torchaudio.save(tmp, chunk, self.sr)

            words = self.align(tmp, seg.text, offset=seg.s)
            os.remove(tmp)

            if words:
                seg.words = words
                # 用 alignment 結果修正 seg 的起止時間
                seg.s = words[0].s
                seg.e = words[-1].e
            aligned_segs.append(seg)

        return aligned_segs

# === ASR ===
class ASR:
    def __init__(self, dev, dt, bs):
        self.dev,self.dt,self.bs = dev,dt,bs
        self.model=None; self._ok=False; self._st='初始化'
        self._mnt=False; self._tag='<|audioplaceholder|>'

    def load(self):
        if self._ok: return self
        self._st='下載模型中...'
        from nemo.collections.speechlm2.models import SALM
        self.model = SALM.from_pretrained('nvidia/canary-qwen-2.5b')
        if self.dev!='cpu': self.model=self.model.to(device=self.dev,dtype=self.dt)
        self.model.eval()
        self._mnt = 'max_new_tokens' in list(inspect.signature(self.model.generate).parameters)
        if hasattr(self.model,'audio_locator_tag'): self._tag=self.model.audio_locator_tag
        self._ok=True; self._st='就緒'
        return self

    @property
    def ready(self): return self._ok
    @property
    def status(self): return self._st

    @torch.inference_mode()
    def _batch(self, paths):
        # 構建 prompt (content 格式)
        prompts = [
            [{'role':'user',
              'content':f'Transcribe the following: {self._tag}',
              'audio':[p]}]
            for p in paths
        ]
        # 嘗試順序與 v3 完全一致：先 max_new_tokens → max_length → bare → slots 變體
        try:
            if self._mnt:
                ids = self.model.generate(prompts=prompts, max_new_tokens=512)
            else:
                try:
                    ids = self.model.generate(prompts=prompts, max_length=512)
                except TypeError:
                    ids = self.model.generate(prompts=prompts)
        except (AssertionError, ValueError) as e1:
            # Fallback 1: slots {'message': ...}
            try:
                ps2 = [[{'role':'user','slots':{'message':f'Transcribe the following: {self._tag}'},'audio':[p]}] for p in paths]
                ids = self.model.generate(prompts=ps2)
            except (AssertionError, ValueError) as e2:
                # Fallback 2: slots {'text': ...}
                try:
                    ps3 = [[{'role':'user','slots':{'text':f'Transcribe the following: {self._tag}'},'audio':[p]}] for p in paths]
                    ids = self.model.generate(prompts=ps3)
                except Exception as e3:
                    raise RuntimeError(
                        f'所有 prompt 格式都失敗:\n'
                        f'  content 格式: {type(e1).__name__}: {e1}\n'
                        f'  slots/message: {type(e2).__name__}: {e2}\n'
                        f'  slots/text:    {type(e3).__name__}: {e3}'
                    )
        return [self.model.tokenizer.ids_to_text(a.cpu()).strip() for a in ids]

    @torch.inference_mode()
    def run(self, wav, dur):
        t0=time.time(); chunks=split_audio(wav,dur); n=len(chunks)
        alive=True
        def tick():
            while alive:
                e=int(time.time()-t0); print(f'\r   ⏳ {e//60:02d}:{e%60:02d}',end='',flush=True); time.sleep(1)
        th=threading.Thread(target=tick,daemon=True); th.start()
        segs,tmps=[],set()
        _first_error_shown = False
        try:
            for i in range(0,n,self.bs):
                b=chunks[i:i+self.bs]; bp=[c[0] for c in b]
                tmps.update(p for p in bp if p!=wav)
                try:
                    tx=self._batch(bp)
                except Exception as batch_err:
                    if not _first_error_shown:
                        print(f'\n   ⚠️ 批次轉錄失敗: {type(batch_err).__name__}: {batch_err}')
                        _first_error_shown = True
                    tx=[]
                    for p in bp:
                        try: tx.append(self._batch([p])[0])
                        except Exception as single_err:
                            if not _first_error_shown:
                                print(f'\n   ⚠️ 單筆轉錄失敗: {type(single_err).__name__}: {single_err}')
                                _first_error_shown = True
                            tx.append('')
                for j,t in enumerate(tx):
                    t=t.strip()
                    if t:
                        if segs: t=dedup(segs[-1].text,t)
                        if t: segs.append(Seg(b[j][1],b[j][2],t))
                for p in bp:
                    if p!=wav:
                        try: os.remove(p); tmps.discard(p)
                        except: pass
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        finally:
            alive=False; th.join(timeout=2); print('\r'+' '*30+'\r',end='')
            for p in tmps:
                try: os.remove(p)
                except: pass
        el=time.time()-t0; rtf=dur/el if el>0 else 0
        return segs, el, rtf

# === Main ===
def main():
    print('🎤 Canary-Qwen-2.5B 轉錄器\n')
    wav=None; asr=None; pool=None
    try:
        if not torch.cuda.is_available():
            print('⚠️ 無 GPU'); dev,dt,bs='cpu',torch.float32,1
        else:
            name=torch.cuda.get_device_name(0)
            vram=torch.cuda.get_device_properties(0).total_memory/(1024**3)
            bs=16 if vram>=38 else (8 if vram>=22 else 4)
            dt=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            dev='cuda:0'
            print(f'🖥️ {name} ({vram:.0f}GB) | batch={bs}')

        asr=ASR(dev,dt,bs)
        pool=ThreadPoolExecutor(max_workers=2)
        fut=pool.submit(asr.load)

        print('📂 請上傳英語音檔：\n')
        if IN_COLAB:
            uploaded=files.upload()
            if not uploaded: return
            src=list(uploaded.keys())[0]
        else:
            src=input('路徑: ')
            if not os.path.exists(src): return

        wav,dur=convert_audio(src)
        print(f'📎 {src} ({dur:.1f}s)')

        if not asr.ready:
            print('⏳ 載入模型...',end='',flush=True)
            while not asr.ready: time.sleep(1)
            print(' ✅')
        fut.result()  # 如果載入失敗，這裡會拋出異常

        segs,el,rtf = asr.run(wav,dur)
        print(f'✅ {el:.1f}s ({rtf:.1f}x RTFx)\n')

        if not segs:
            print('⚠️ 無轉錄結果')
            print('   如果上方沒有顯示錯誤，可能是音檔太短或內容為靜音')
            return

        # CTC Forced Alignment
        print('🎯 CTC Forced Alignment...',end='',flush=True)
        try:
            aligner = CTCAligner(device=dev)
            segs = aligner.align_segments(wav, segs)
            n_words = sum(len(s.words) for s in segs if s.words)
            print(f' ✅ ({n_words} words)')
        except Exception as ae:
            print(f' ⚠️ 跳過 ({ae})')

        bd='/content' if IN_COLAB else os.getcwd()
        bn=os.path.splitext(os.path.basename(src))[0]

        # TXT: 逐段 + 逐字時間戳
        tp=os.path.join(bd,f'{bn}_transcript.txt')
        with open(tp,'w',encoding='utf-8') as f:
            for s in segs:
                f.write(s.line()+'\n')
                if s.words:
                    for w in s.words:
                        f.write(f'  [{_fmt(w.s)} → {_fmt(w.e)}] {w.text}\n')

        # SRT: word-level (每個 word 一條字幕)
        sp=os.path.join(bd,f'{bn}_subtitle.srt')
        with open(sp,'w',encoding='utf-8') as f:
            idx=1
            for s in segs:
                if s.words:
                    # 每 ~5 個 word 合併成一條字幕
                    for g in range(0, len(s.words), 5):
                        group = s.words[g:g+5]
                        gs = group[0].s; ge = group[-1].e
                        gt = ' '.join(w.text for w in group)
                        f.write(f'{idx}\n{_fmt(gs,True)} --> {_fmt(ge,True)}\n{gt}\n\n')
                        idx += 1
                else:
                    f.write(s.srt(idx)+'\n')
                    idx += 1

        # 預覽
        for s in segs[:10]:
            print(s.line())
            if s.words:
                wt = ' | '.join(f'{w.text}({_fmt(w.s)})' for w in s.words[:8])
                if len(s.words)>8: wt += f' ... +{len(s.words)-8}'
                print(f'  └─ {wt}')
        if len(segs)>10: print(f'... 共 {len(segs)} 段')
        print(f'\n💾 {tp}\n💾 {sp}')
        if IN_COLAB: files.download(tp); files.download(sp)

    except Exception as e:
        print(f'\n❌ {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()
    finally:
        if wav and os.path.exists(wav):
            try: os.remove(wav)
            except: pass
        if pool: pool.shutdown(wait=False)
        if asr and asr.model: del asr.model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()

main()
